# RAG pipeline with python
Feeding a whole 50-page PDF into an LLM costs too much money and hits token limits fast. Even if it fits, the model often misses the specific answer because it gets lost in a sea of irrelevant text. We need a way to give the model only the exact paragraphs it needs to answer the question.

To fix this, we use a technique called **chunking**. This basically means we chop your large documents into smaller, bite-sized pieces before saving them to a database. By breaking a book into 500-token pieces, we can search for the most relevant "chunks" and send only those to the AI. This makes your RAG pipeline faster, cheaper, and much more accurate.

The most common way to do this is using a recursive character splitter. It tries to keep paragraphs and sentences together so the text still makes sense. You can set a chunk_size for the length and a chunk_overlap so no context gets cut off at the edges.



In [ ]:
pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# We want 500 characters per chunk
# We keep 50 characters from the previous chunk so sentences don't break
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Sample document text
large_document_string = """
LangChain is a robust framework for developing LLM-powered applications. It excels at managing context.

Retrieval-Augmented Generation (RAG) is a highly effective pattern. It injects relevant data right into the model prompt to minimize hallucinations.
"""

chunks = splitter.split_text(large_document_string)
print(f"Created {len(chunks)} chunks from the file.")

In [ ]:
!pip install ipywidgets

# Langchain Retrieval Chain (FOR RAG)

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Get API key from environment
api_key = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI(api_key=api_key)

# Test call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"Explain this Python error: IndexError"}]
)

print(response.choices[0].message.content)


In [ ]:
import os
from dotenv import load_dotenv

# 1. Force reload the .env file explicitly
# Replace the path below with your actual root path if needed
load_dotenv(dotenv_path=r'C:\Users\lrame\code\ai_agent_labs\.env', override=True)

# 2. DEBUG: Check if it actually exists in this Python process
key = os.environ.get("OPENAI_API_KEY")
print(f"Is API key present? {key is not None}")
if key:
    print(f"Key starts with: {key[:5]}...") # Just printing the start to verify

# 3. Initialize the LLM only after verification
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")
print("LLM initialized successfully!")

In [ ]:
import sys
!{sys.executable} -m pip install langchain langchain-openai langchain-community faiss-cpu python-dotenv

In [ ]:
import pkg_resources

packages = ["langchain", "langchain-openai", "langchain-community", "faiss-cpu", "python-dotenv"]
for package in packages:
    try:
        dist = pkg_resources.get_distribution(package)
        print(f"{package} is installed: {dist.version}")
    except pkg_resources.DistributionNotFound:
        print(f"{package} is NOT installed.")

In [ ]:
import sys
# This lists all the directories where Python looks for modules
print(sys.path)

In [ ]:
!pip install langchain-core

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Force load the .env file with override to ensure it's captured
load_dotenv(override=True)

# 1. Prepare your components
embeddings = OpenAIEmbeddings()
my_vector_store = FAISS.from_texts(["To reset your password, click settings."], embeddings)
llm = ChatOpenAI(model="gpt-3.5-turbo")

# 2. Define the Prompt Template
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Build the LCEL Chain
chain = (
    {"context": my_vector_store.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Invoke the Chain
response = chain.invoke("How do I reset my password?")
print(response)